# 07 · BERTopic sobre el subcorpus COM-B

Modelado de tópicos sobre los tweets que expresan componentes del COM-B, reutilizando los embeddings del NB03. Equivalente a `9_bertopic_chunks_ciencia.ipynb` de Karen.

In [1]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosCOMB')

NR_TOPICS_GRID  = [5, 10, 15, 20, "auto"]
NR_TOPICS_FINAL = 10
MIN_TOPIC_SIZE  = 5

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


[CONFIG] OK
  DATA_PROCESSED : C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosCOMB


In [2]:
# ============================================================
# CELL 0b - COMPATIBILIDAD NUMPY 2.x / TENSORFLOW
# ============================================================
import sys, types, importlib.machinery

if 'tensorflow' not in sys.modules:
    def _fake_mod(name):
        m = types.ModuleType(name)
        m.__spec__    = importlib.machinery.ModuleSpec(name, loader=None)
        m.__path__    = []
        m.__package__ = name
        m.__version__ = '0.0.0'
        return m
    for _n in [
        'tensorflow', 'tensorflow.python', 'tensorflow.python.eager',
        'tensorflow.python.framework', 'tensorflow.python.client',
        'tensorflow.python.util', 'tensorflow.python.ops',
        'tensorflow.core', 'tensorflow.keras', 'tensorflow.keras.layers',
        'tensorflow.keras.models', 'tensorflow.keras.callbacks',
        'tensorflow.keras.losses', 'tensorflow.keras.optimizers',
        'tensorflow.compat', 'tensorflow.compat.v1', 'tensorflow.compat.v2',
    ]:
        sys.modules[_n] = _fake_mod(_n)
    print('[COMPAT] tensorflow reemplazado con modulos ficticios')
else:
    print('[COMPAT] tensorflow ya estaba cargado, sin cambios')


[COMPAT] tensorflow reemplazado con modulos ficticios


In [3]:
# ============================================================
# CELL 1 - IMPORTS Y CARGA
# ============================================================
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

tweets_etiquetados = pd.read_parquet(DATA_PROCESSED / 'comb_etiquetados_final.parquet')

tweets_comp  = tweets_etiquetados[tweets_etiquetados['etiqueta_comb'] == 1].copy()

df_comp  = tweets_comp.merge(
    corpus[['id_doc', 'Texto_limpio']],
    on='id_doc', how='left'
)

print(f'Corpus completo    : {len(corpus):,} tweets')
print(f'Subcorpus comp.    : {len(df_comp):,} tweets')
df_comp.head(2)


Corpus completo    : 151,424 tweets
Subcorpus comp.    : 12,254 tweets


,chunk_id,id_doc,texto_chunk,COMB_Capacidad_fisica,COMB_Capacidad_psicologica,COMB_Motivacion_automatica,COMB_Motivacion_reflexiva,COMB_Oportunidad_fisica,COMB_Oportunidad_social,score_max,subcat_max,etiqueta_comb,categoria_detectada,Texto_limpio
0,5_1,5,Local Gestionan regidores cajas para intubació...,0.374892,0.315076,0.261424,0.284109,0.510342,0.272318,0.510342,COMB_Oportunidad_fisica,1,COMB_Oportunidad_fisica,Local Gestionan regidores cajas para intubació...
1,6_1,6,La otra semana ya saldrá el aumento de contagi...,0.281226,0.367581,0.299257,0.363866,0.482989,0.208974,0.482989,COMB_Oportunidad_fisica,1,COMB_Oportunidad_fisica,La otra semana ya saldrá el aumento de contagi...


In [4]:
# ============================================================
# CELL 2 - CARGAR Y FILTRAR EMBEDDINGS
# ============================================================
embeddings_all = np.load(DATA_PROCESSED / 'tweet_embeddings.npy')
print(f'Embeddings totales  : {embeddings_all.shape}')

mask_comp        = tweets_etiquetados['etiqueta_comb'] == 1
embeddings_comp  = embeddings_all[mask_comp.values]

print(f'Embeddings del COM-B : {embeddings_comp.shape}')
assert len(embeddings_comp) == len(df_comp), (
    f"Mismatch: {len(embeddings_comp)} embeddings vs {len(df_comp)} tweets"
)
print('OK - embeddings alineados correctamente')


Embeddings totales  : (151424, 768)
Embeddings del COM-B : (12254, 768)
OK - embeddings alineados correctamente


In [5]:
# ============================================================
# CELL 3 - STOPWORDS Y VECTORIZADOR
# ============================================================
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
spanish_stopwords = list(nlp.Defaults.stop_words)

twitter_sw = [
    'rt', 'http', 'https', 'co', 'amp', 'via', 'q', 'xq', 'x',
    'si', 'ya', 'asi', 'tan', 'ser', 'hay', 'ver', 'hoy',
]
all_stopwords = list(set(spanish_stopwords + twitter_sw))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    max_features=5000,
    min_df=2,
    ngram_range=(1, 2),
    strip_accents=None
)

documents = df_comp['Texto_limpio'].fillna('').tolist()
print(f'Documentos para BERTopic: {len(documents):,}')


Documentos para BERTopic: 12,254


In [6]:
# ============================================================
# CELL 4 - FUNCIONES DE COHERENCIA
# ============================================================

def get_topic_words(topic_model, top_n=10):
    topics_words = []
    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)
    return topics_words


def compute_bertopic_coherence(topic_model, documents, vectorizer_model, top_n_words=10):
    analyzer       = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]
    dictionary     = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)
    corpus_bow     = [dictionary.doc2bow(doc) for doc in tokenized_docs]
    topics_words   = get_topic_words(topic_model, top_n=top_n_words)
    cm = CoherenceModel(
        topics=topics_words, texts=tokenized_docs,
        dictionary=dictionary, corpus=corpus_bow, coherence='c_v'
    )
    return cm.get_coherence(), cm.get_coherence_per_topic()


In [7]:
# ============================================================
# CELL 5 - BUSQUEDA EN GRILLA DE nr_topics
# ============================================================
results = []

for nr in NR_TOPICS_GRID:
    print(f'\n--- nr_topics={nr} ---')
    tm = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        verbose=False,
        language="spanish"
    )
    topics, _ = tm.fit_transform(documents, embeddings=embeddings_comp)
    new_topics = tm.reduce_outliers(
        documents, topics, strategy="c-tf-idf", embeddings=embeddings_comp
    )
    tm.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence, _ = compute_bertopic_coherence(tm, documents, vectorizer_model)
    n_outliers   = sum(1 for t in tm.topics_ if t == -1)
    results.append({
        "nr_topics"     : nr,
        "n_topics_final": len([t for t in tm.get_topics() if t != -1]),
        "coherence_c_v" : coherence,
        "pct_outliers"  : n_outliers / len(documents)
    })
    r = results[-1]
    print(f'  Topicos: {r["n_topics_final"]}  Coh: {r["coherence_c_v"]:.4f}  Outliers: {r["pct_outliers"]:.2%}')

df_grid = pd.DataFrame(results)
print()
print(df_grid.sort_values("coherence_c_v", ascending=False).to_string(index=False))



--- nr_topics=5 ---


2026-07-14 07:30:47,684 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 4  Coh: 0.5374  Outliers: 0.00%

--- nr_topics=10 ---


2026-07-14 07:31:03,356 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 9  Coh: 0.5442  Outliers: 0.01%

--- nr_topics=15 ---


...

  Topicos: 14  Coh: 0.5177  Outliers: 0.01%

--- nr_topics=20 ---


2026-07-14 07:31:39,612 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 19  Coh: 0.4962  Outliers: 0.00%

--- nr_topics=auto ---


2026-07-14 07:31:56,393 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 58  Coh: 0.4457  Outliers: 0.00%

nr_topics  n_topics_final  coherence_c_v  pct_outliers
       10               9       0.544247      0.000082
        5               4       0.537415      0.000000
       15              14       0.517650      0.000082
       20              19       0.496203      0.000000
     auto              58       0.445726      0.000000


In [8]:
# ============================================================
# CELL 6 - MODELO FINAL
# ============================================================
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=NR_TOPICS_FINAL,
    min_topic_size=MIN_TOPIC_SIZE,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)

topics, probs = topic_model.fit_transform(documents, embeddings=embeddings_comp)
new_topics    = topic_model.reduce_outliers(
    documents, topics, strategy="c-tf-idf", embeddings=embeddings_comp
)
topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

tm_res = topic_model.get_topic_info()
print(tm_res.head(10))


2026-07-14 07:32:10,010 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-14 07:32:11,642 - BERTopic - Dimensionality - Completed ✓
2026-07-14 07:32:11,644 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-14 07:32:12,019 - BERTopic - Cluster - Completed ✓
2026-07-14 07:32:12,020 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-14 07:32:12,722 - BERTopic - Representation - Completed ✓
2026-07-14 07:32:12,723 - BERTopic - Topic reduction - Reducing number of topics
2026-07-14 07:32:12,734 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-14 07:32:13,352 - BERTopic - Representation - Completed ✓
2026-07-14 07:32:13,356 - BERTopic - Topic reduction - Reduced number of topics from 251 to 10
2026-07-14 07:32:13,587 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure th

   Topic  Count                                               Name  \
0      0   9600                    0_covid19_pandemia_vacuna_salud   
1      1   1644                           1_gente_covid19_covid_19   
2      2    545             2_covid19_síntomas_paciente_enfermedad   
3      3    170                  3_información_covid19_falta_cosas   
4      4     67                  4_actividad_física_mental_covid19   
5      5     97  5_protocolos bioseguridad_bioseguridad_protoco...   
6      6     42  6_sospecha_sospecha covid19_modelo_pacientes s...   
7      7     50                     7_oms_dra_discapacidad_ginebra   
8      8     39                   8_fiebre_tos_descarga_aplicación   

                                      Representation  \
0  [covid19, pandemia, vacuna, salud, virus, coro...   
1  [gente, covid19, covid, 19, covid 19, casa, cu...   
2  [covid19, síntomas, paciente, enfermedad, posi...   
3  [información, covid19, falta, cosas, conocimie...   
4  [actividad, físi

In [9]:
# ============================================================
# CELL 7 - TABLA DE RESULTADOS FINAL
# ============================================================
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
print(f'Coherencia global (c_v): {coh_global:.4f}')

topic_ids    = [t for t in topic_model.get_topics() if t != -1]
coherence_df = pd.DataFrame({"Topic": topic_ids, "Coherence_c_v": coh_per_topic})

total_docs  = tm_res["Count"].sum()
top         = tm_res.sort_values("Count", ascending=False).head(20).copy()
top["Keywords"]   = top["Representation"].apply(lambda ws: ", ".join(ws))
top["Porcentaje"] = (top["Count"] / total_docs * 100).round(2)
top = top.merge(coherence_df, on="Topic", how="left")

final_table = top[["Topic", "Count", "Porcentaje", "Coherence_c_v", "Keywords"]]
pd.set_option("display.max_colwidth", None)
print(final_table.to_string(index=False))


Coherencia global (c_v): 0.5307
 Topic  Count  Porcentaje  Coherence_c_v                                                                                                                         Keywords
     0   9600       78.34       0.178580                                  covid19, pandemia, vacuna, salud, virus, coronavirus, pacientes, contagio, hospitales, personas
     1   1644       13.42       0.364339                                               gente, covid19, covid, 19, covid 19, casa, cuarentena, quedateencasa, salir, calle
     2    545        4.45       0.430454                            covid19, síntomas, paciente, enfermedad, positivo, prueba, recuperación, alta, años, positivo covid19
     3    170        1.39       0.337195                                       información, covid19, falta, cosas, conocimiento, entender, expertos, decisiones, has, ojo
     5     97        0.79       0.535767                    protocolos bioseguridad, bioseguridad, protocolos, cita, c

In [10]:
# ============================================================
# CELL 8 - ASIGNAR TOPICO Y GUARDAR
# ============================================================
import pickle

df_comp = df_comp.copy()
df_comp['topico'] = topic_model.topics_

coherence_table = (
    tm_res[tm_res["Topic"] != -1].merge(coherence_df, on="Topic", how="left")
)

ruta_excel = DATA_PROCESSED / 'bertopic_comb_resultados.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    final_table.to_excel(writer,     sheet_name='Top_topicos',          index=False)
    coherence_table.to_excel(writer, sheet_name='Coherencia_por_topico',index=False)
    df_grid.to_excel(writer,         sheet_name='Grid_nr_topics',       index=False)
    df_comp[['id_doc', 'topico', 'categoria_detectada']].to_excel(
        writer, sheet_name='Tweets_por_topico', index=False
    )
print('[GUARDADO] bertopic_comb_resultados.xlsx')

df_comp.to_parquet(DATA_PROCESSED / 'comb_tweets_bertopic.parquet', index=False)
print('[GUARDADO] comb_tweets_bertopic.parquet')

print()
print('Notebook 07 completado.')
print('Siguiente -> 08_extraer_frecuencias_POS.ipynb')


[GUARDADO] bertopic_comb_resultados.xlsx
[GUARDADO] comb_tweets_bertopic.parquet

Notebook 07 completado.
Siguiente -> 08_extraer_frecuencias_POS.ipynb
